#### Conclusions
- Overfit on training
- Small accuracy
- Too many epochs
- Small image size
- Primitive architecture

In [1]:
import os
import glob
import hashlib
import random
import numpy as np
import pandas as pd
import shutil
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, random_split
from torchvision import transforms
import torch.optim as optim
from torchvision import models

In [2]:
print('ebadi: ', torch.cuda.is_available())
print('dranduliator: ', torch.cuda.get_device_name(0))

ebadi:  True
dranduliator:  NVIDIA GeForce GTX 1050


In [3]:
path = 'datasets/stanford_dogs'
imgs_dir = os.path.join(path, 'Images')

# matrix properties
img_size=128
batch_size=8
num_classes=170
transform = transforms.Compose([
    transforms.Resize((img_size,img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

class Dataset(Dataset):
    def __init__(self, imgs, lbls, transform=None):
        self.imgs=imgs
        self.lbls=lbls
        self.transform=transform
    def __len__(self):
        return len(self.imgs)
    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]).convert('RGB')
        img = self.transform(img)
        return img, self.lbls[idx]

sub_imgs=[]
sub_lbls=[]
classes = sorted(os.listdir(imgs_dir))
class_to_idx = {cls: i for i, cls in enumerate(classes)}
for cls, idx in class_to_idx.items():
    img_path = sorted(glob.glob(os.path.join(imgs_dir,cls, '*jpg')))
    img_path = img_path[:num_classes]
    sub_imgs.extend(img_path)
    sub_lbls.extend([idx]*len(img_path))
perm = np.random.permutation(len(sub_imgs))
sub_imgs=[sub_imgs[i] for i in perm]
sub_lbls=[sub_lbls[i] for i in perm]

dset = Dataset(sub_imgs, sub_lbls, transform=transform)
n_total = len(dset)
n_train = int(0.8*n_total)
n_val = int(0.1*n_total)
n_test = n_total - n_train-n_val
# split
ds_tr, ds_val, ds_te = random_split(dset, [n_train, n_val, n_test])
tr_loader = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(ds_val, batch_size=batch_size, shuffle=False, num_workers=2)
te_loader = DataLoader(ds_te, batch_size=batch_size, shuffle=False, num_workers=2)

In [4]:
print(f"Train batches: {len(tr_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(te_loader)}")

Train batches: 4098
Val batches:   513
Test batches:  513


In [5]:
device = torch.device('cuda')
# cnn arch
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()
        self.features = nn.Sequential(
            # inp 128x3, kern 3, pool 2, relu
            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2,2),
            # inp 64x64, kern 3, pool 2, relu
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2,2),
            # 32X128, kern 3, pool2, relu
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2,2),
            # 16x256, kern 3, pool2, relu
            nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2,2),
            # 8x256, kern3, pool2, relu
            nn.Conv2d(512, 256 kernel_size=3, padding=1), nn.BatchNorm2d(5), nn.ReLU(), nn.MaxPool2d(2,2)
        )
        self.classifier = nn.Sequential(
            # 4x512 -> 512, relu, drop 0.5 -> 120, softmax in crossentropy
            nn.Flatten(), nn.Linear(8*8*512, 512), nn.ReLU(), nn.Dropout(0.5), nn.Linear(512, num_classes)           
        )
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

In [45]:
#  MODEL HERE, jogaltip alma
model = CNN(num_classes=num_classes).to(device)
# loss, acc, optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=10**-4)
tr_loss, val_loss, te_loss = [], [], []
tr_acc, val_acc, te_acc = [], [], []
for e in range(15):
    # train
    model.train()
    running_loss, correct, total = 0,0,0

    for imgs, lbls in tr_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        output = model(imgs)
        loss = criterion(output, lbls)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()*imgs.size(0)
        _,pred = output.max(1)
        total += lbls.size(0)
        correct += pred.eq(lbls).sum().item()
    tr_loss.append(running_loss/total)
    tr_acc.append(correct/total)
    #validation
    model.eval()
    running_loss, correct, total = 0,0,0

    for imgs, lbls in val_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        
        output = model(imgs)
        loss = criterion(output, lbls)

        running_loss += loss.item()*imgs.size(0)
        _,pred = output.max(1)
        total += lbls.size(0)
        correct += pred.eq(lbls).sum().item()
    val_loss.append(running_loss/total)
    val_acc.append(correct/total)
    #test
    running_loss, correct, total = 0,0,0
    
    with torch.no_grad():
        for imgs, lbls in te_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            
            output = model(imgs)
            loss = criterion(output, lbls)
    
            running_loss += loss.item()*imgs.size(0)
            _,pred = output.max(1)
            total += lbls.size(0)
            correct += pred.eq(lbls).sum().item()
        te_loss.append(running_loss/total)
        te_acc.append(correct/total)
    print(f"epoch {e+1} | "
        f"train Acc: {tr_acc[-1]:.3f}, val acc: {val_acc[-1]:.3f}, test acc: {te_acc[-1]:.3f}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacity of 3.94 GiB of which 3.69 MiB is free. Including non-PyTorch memory, this process has 3.93 GiB memory in use. Of the allocated memory 3.80 GiB is allocated by PyTorch, and 64.69 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)